# Process mooring files and make them xarray-friendly

In [1]:
%matplotlib inline
%config InlineBackend.figure_format='retina'

import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import gridspec
import matplotlib.path as mpath
from matplotlib import ticker, cm
import numpy as np
import netCDF4 as nc
import xarray as xr
import cmocean.cm as cmocean
import glob,os, re
import matplotlib.colors as mcolors
from mpl_toolkits.axes_grid1 import make_axes_locatable
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import gsw, intake

import cartopy.crs as ccrs
import cartopy.feature as cfeature

import logging
logging.captureWarnings(True)
logging.getLogger('py.warnings').setLevel(logging.ERROR)


from dask.distributed import Client
import dask.distributed as dask

figdir = '/g/data/x77/ps7863/figures/AABW_variability/'
degree = u'\N{DEGREE SIGN}'

In [2]:
from os import environ
environ["PYTHONWARNINGS"] = "ignore"

In [3]:
client = Client(threads_per_worker=1,memory_limit=0)
client.amm.start()
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /proxy/37247/status,
Dashboard: /proxy/37247/status,Workers: 24
Total threads: 24,Total memory: 0 B
Status: running,Using processes: True
Comm: tcp://127.0.0.1:41491,Workers: 0
Dashboard: /proxy/37247/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:36459,Total threads: 1
Dashboard: /proxy/38647/status,Memory: 0 B
Nanny: tcp://127.0.0.1:35847,


In [4]:
iaf_cycle3 = '01deg_jra55v140_iaf_cycle3'
catalog = intake.cat.access_nri

In [5]:
Weddell_moorings={
    'AWI217_1990': {'lon': -45.85, 'lat': -64.4183},
    'AWI217_1993': {'lon': -45.8305, 'lat': -64.4168},
    'AWI217_2008': {'lon': -45.873, 'lat': -64.3938},
    'AWI217_2011': {'lon': -45.8658, 'lat': -64.398},
    'AWI217_2013': {'lon': -45.8687, 'lat': -64.3823}
}

In [10]:
files = [
    '/g/data/x77/ps7863/data/AABW_variability/moorings/AWI217_1990.nc',
    '/g/data/x77/ps7863/data/AABW_variability/moorings/AWI217_1993.nc',
    '/g/data/x77/ps7863/data/AABW_variability/moorings/AWI217_2008.nc',
    '/g/data/x77/ps7863/data/AABW_variability/moorings/AWI217_2011.nc', 
    '/g/data/x77/ps7863/data/AABW_variability/moorings/AWI217_2013.nc'
]

In [7]:
def process_mooring_files_deepest_only(
    files,
    mooring_meta=None,
    out_dir="/g/data/x77/ps7863/data/AABW_variability/moorings/",
    mooring_name_strategy="infer_from_filename",
    filename_mooring_regex=r"([A-Za-z0-9]+_\d{4})",
    overwrite=False,
    depth_stat="median",  # "median" or "max"
):
    """
    For each file, select ONLY the deepest instrument (highest median/max pressure)
    among instruments that have temp+salin+press. If that instrument lacks time,
    use Instrument_01_date/time as a fallback.

    Writes one xarray-friendly .nc per input file.
    """
    os.makedirs(out_dir, exist_ok=True)
    if mooring_meta is None:
        mooring_meta = {}

    # Regex for variables like "Instrument_02_temp"
    pat_temp  = re.compile(r"Instrument_(\d+)_temp$", re.IGNORECASE)
    pat_salin = re.compile(r"Instrument_(\d+)_salin$", re.IGNORECASE)
    pat_press = re.compile(r"Instrument_(\d+)_press$", re.IGNORECASE)

    def time_var_name(ds, inst_id):
        for c in (f"Instrument_{inst_id}_date", f"Instrument_{inst_id}_time"):
            if c in ds.variables:
                return c
        return None

    def time_var_name_inst01(ds):
        for c in ("Instrument_01_date", "Instrument_01_time"):
            if c in ds.variables:
                return c
        return None

    def infer_mooring_name(path, ds):
        base = os.path.basename(path)
        if mooring_name_strategy == "use_file_attr":
            mn = ds.attrs.get("mooring_name")
            if mn:
                return str(mn)
        if mooring_name_strategy in ("use_file_attr", "infer_from_filename"):
            m = re.search(filename_mooring_regex, base)
            if m:
                return m.group(1)
        return os.path.splitext(base)[0] if mooring_name_strategy != "none" else None

    def depth_metric(arr):
        if depth_stat.lower() == "max":
            return np.nanmax(arr)
        return np.nanmedian(arr)

    summary = {"processed": [], "skipped": [], "chosen_instrument": {}}

    for f in files:
        try:
            with xr.open_dataset(f, decode_times=True) as ds:
                mooring_name = infer_mooring_name(f, ds)

                # find instrument IDs that have temp+salin+press
                temps  = {pat_temp.match(v).group(1)  for v in ds.variables if pat_temp.match(v)}
                salins = {pat_salin.match(v).group(1) for v in ds.variables if pat_salin.match(v)}
                presss = {pat_press.match(v).group(1) for v in ds.variables if pat_press.match(v)}
                candidates = sorted(temps & salins & presss, key=lambda x: int(x))

                if not candidates:
                    summary["skipped"].append({"file": f, "reason": "No instrument with temp+salin+press"})
                    continue

                # choose deepest by median/max pressure (ignoring NaNs)
                best_inst = None
                best_metric = -np.inf
                for inst in candidates:
                    pvar = f"Instrument_{inst}_press"
                    try:
                        p = np.asarray(ds[pvar]).squeeze()
                        m = depth_metric(p)
                        if np.isfinite(m) and m > best_metric:
                            best_metric = m
                            best_inst = inst
                    except Exception:
                        continue

                if best_inst is None:
                    summary["skipped"].append({"file": f, "reason": "Could not compute depth metric"})
                    continue

                summary["chosen_instrument"][f] = int(best_inst)

                # pull required variables for chosen instrument
                tvar = f"Instrument_{best_inst}_temp"
                svar = f"Instrument_{best_inst}_salin"
                pvar = f"Instrument_{best_inst}_press"
                dvar = time_var_name(ds, best_inst)

                temp = np.squeeze(ds[tvar].data)
                salt = np.squeeze(ds[svar].data)
                pres = np.squeeze(ds[pvar].data)

                # time from chosen instrument or fallback to Instrument_01
                if dvar is None:
                    dvar_01 = time_var_name_inst01(ds)
                    if dvar_01 is None:
                        summary["skipped"].append({"file": f, "reason": f"No time for instrument {best_inst} and no Instrument_01 time"})
                        continue
                    time = np.squeeze(ds[dvar_01].data)
                else:
                    time = np.squeeze(ds[dvar].data)

                # basic shape check
                if time.ndim != 1:
                    time = time.reshape(-1)
                n = len(time)
                if any(arr.ndim != 1 or len(arr) != n for arr in (temp, salt, pres)):
                    summary["skipped"].append({"file": f, "reason": f"Length mismatch between time and variables for inst {best_inst}"})
                    continue

                # lon/lat/bottom_depth from file first, else mooring_meta
                def _maybe_scalar(da, names):
                    for nm in names:
                        if nm in ds:
                            try:
                                return float(ds[nm].item())
                            except Exception:
                                pass
                    return None
                file_lon = _maybe_scalar(ds, ("lon","longitude","LON"))
                file_lat = _maybe_scalar(ds, ("lat","latitude","LAT"))
                bottom_depth = _maybe_scalar(ds, ("bottom_depth",))

                meta_lon = meta_lat = None
                if mooring_name and mooring_name in mooring_meta:
                    meta_lon = mooring_meta[mooring_name].get("lon")
                    meta_lat = mooring_meta[mooring_name].get("lat")

                use_lon = file_lon if file_lon is not None else meta_lon
                use_lat = file_lat if file_lat is not None else meta_lat

                # build dataset
                da_temp = xr.DataArray(temp, dims=["time"], coords={"time": time}, name="temperature")
                da_salt = xr.DataArray(salt, dims=["time"], coords={"time": time}, name="salinity")
                da_pres = xr.DataArray(pres, dims=["time"], coords={"time": time}, name="pressure")
                ds_out = xr.Dataset({"temperature": da_temp, "salinity": da_salt, "pressure": da_pres})

                if use_lon is not None:
                    ds_out = ds_out.assign_coords(lon=float(use_lon))
                if use_lat is not None:
                    ds_out = ds_out.assign_coords(lat=float(use_lat))

                # attrs
                def _fmt_date(x):
                    try:
                        return np.array(x).astype("datetime64[ns]").astype("datetime64[D]").astype(str)
                    except Exception:
                        return str(x)
                start_date = _fmt_date(time[0])
                end_date   = _fmt_date(time[-1])

                description = f"Data from mooring {mooring_name}, deepest instrument (by {depth_stat})" if mooring_name else "Processed mooring instrument data (deepest)"
                ds_out = ds_out.assign_attrs(
                    Description=description,
                    bottom_depth=bottom_depth if bottom_depth is not None else np.nan,
                    lon=float(use_lon) if use_lon is not None else np.nan,
                    lat=float(use_lat) if use_lat is not None else np.nan,
                    start_date=start_date,
                    end_date=end_date,
                    source_file=os.path.basename(f),
                    instrument_choice="deepest",
                    instrument_id=int(best_inst),
                )

                # TEOS-10: SA & CT (only if lon/lat known)
                if (use_lon is not None) and (use_lat is not None):
                    SA = gsw.SA_from_SP(ds_out["salinity"], ds_out["pressure"], ds_out.lon, ds_out.lat)
                    CT = gsw.CT_from_t(SA, ds_out["temperature"], ds_out["pressure"])
                    ds_out["SA"] = SA
                    ds_out["CT"] = CT

                # write one file per input
                base = mooring_name or os.path.splitext(os.path.basename(f))[0]
                out_path = os.path.join(out_dir, f"{base}_deepest.nc")
                if (not overwrite) and os.path.exists(out_path):
                    summary["skipped"].append({"file": out_path, "reason": "Exists (overwrite=False)"})
                else:
                    ds_out.to_netcdf(out_path)
                    summary["processed"].append(out_path)

        except Exception as e:
            summary["skipped"].append({"file": f, "reason": f"Error: {e}"})

    return summary


In [9]:
process_mooring_files_deepest_only(files,
                      mooring_meta=Weddell_moorings,
                    out_dir='/g/data/x77/ps7863/data/AABW_variability/moorings/processed/',
                    mooring_name_strategy="infer_from_filename",  # or "use_file_attr" if your files have ds.attrs["mooring_name"]
                    overwrite=False
)

{'processed': ['/g/data/x77/ps7863/data/AABW_variability/moorings/processed/AWI217_1990_deepest.nc',
  '/g/data/x77/ps7863/data/AABW_variability/moorings/processed/AWI217_1993_deepest.nc',
  '/g/data/x77/ps7863/data/AABW_variability/moorings/processed/AWI217_2008_deepest.nc',
  '/g/data/x77/ps7863/data/AABW_variability/moorings/processed/AWI217_2011_deepest.nc',
  '/g/data/x77/ps7863/data/AABW_variability/moorings/processed/AWI217_2013_deepest.nc'],
 'skipped': [],
 'chosen_instrument': {'/g/data/x77/ps7863/data/AABW_variability/moorings/AWI217_1990.nc': 2,
  '/g/data/x77/ps7863/data/AABW_variability/moorings/AWI217_1993.nc': 1,
  '/g/data/x77/ps7863/data/AABW_variability/moorings/AWI217_2008.nc': 1,
  '/g/data/x77/ps7863/data/AABW_variability/moorings/AWI217_2011.nc': 2,
  '/g/data/x77/ps7863/data/AABW_variability/moorings/AWI217_2013.nc': 2}}

In [56]:
files = [
    '/g/data/x77/ps7863/data/AABW_variability/moorings/AWI207_1989.nc',
    '/g/data/x77/ps7863/data/AABW_variability/moorings/AWI207_1990.nc',
    '/g/data/x77/ps7863/data/AABW_variability/moorings/AWI207_1993.nc',
    '/g/data/x77/ps7863/data/AABW_variability/moorings/AWI207_1996.nc',
    '/g/data/x77/ps7863/data/AABW_variability/moorings/AWI207_2005.nc', 
    '/g/data/x77/ps7863/data/AABW_variability/moorings/AWI207_2008.nc', 
    '/g/data/x77/ps7863/data/AABW_variability/moorings/AWI207_2011.nc',
    '/g/data/x77/ps7863/data/AABW_variability/moorings/AWI207_2013.nc', 
    '/g/data/x77/ps7863/data/AABW_variability/moorings/AWI207_2017.nc'
]

In [57]:
process_mooring_files_deepest_only(files,
                    mooring_meta=Weddell_moorings,
                    out_dir='/g/data/x77/ps7863/data/AABW_variability/moorings/processed/',
                    mooring_name_strategy="infer_from_filename",  # or "use_file_attr" if your files have ds.attrs["mooring_name"]
                    overwrite=False
)

{'processed': ['/g/data/x77/ps7863/data/AABW_variability/moorings/processed/AWI207_1989_deepest.nc',
  '/g/data/x77/ps7863/data/AABW_variability/moorings/processed/AWI207_1990_deepest.nc',
  '/g/data/x77/ps7863/data/AABW_variability/moorings/processed/AWI207_1993_deepest.nc',
  '/g/data/x77/ps7863/data/AABW_variability/moorings/processed/AWI207_1996_deepest.nc',
  '/g/data/x77/ps7863/data/AABW_variability/moorings/processed/AWI207_2005_deepest.nc',
  '/g/data/x77/ps7863/data/AABW_variability/moorings/processed/AWI207_2008_deepest.nc',
  '/g/data/x77/ps7863/data/AABW_variability/moorings/processed/AWI207_2011_deepest.nc',
  '/g/data/x77/ps7863/data/AABW_variability/moorings/processed/AWI207_2013_deepest.nc',
  '/g/data/x77/ps7863/data/AABW_variability/moorings/processed/AWI207_2017_deepest.nc'],
 'skipped': [],
 'chosen_instrument': {'/g/data/x77/ps7863/data/AABW_variability/moorings/AWI207_1989.nc': 2,
  '/g/data/x77/ps7863/data/AABW_variability/moorings/AWI207_1990.nc': 1,
  '/g/data/

In [68]:
ds = xr.open_dataset('/g/data/x77/ps7863/data/AABW_variability/moorings/processed/AWI207_2013_deepest.nc')

In [69]:
ds.pressure.mean(), ds.bottom_depth

(<xarray.DataArray 'pressure' ()> Size: 8B
 array(2518.79289974)
 Coordinates:
     lon      float64 8B ...
     lat      float64 8B ...,
 2482.0)